# Multi-Model Inference & Classification

This notebook loads a folder of XISF images, performs full pre-processing, extracts star cutouts, and runs them through 7 champion models (incl. DenseNet/VGG).

In [1]:
# --- Imports ---
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
import numpy as np
import xisf
import cv2
import os
import tkinter as tk
from tkinter import filedialog
from pathlib import Path
from collections import Counter
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
from astropy.stats import sigma_clipped_stats
from photutils.detection import DAOStarFinder
import openpyxl
from openpyxl.styles import PatternFill, Font
from openpyxl.utils import get_column_letter

# --- Config ---
DEFAULT_INPUT_DIR = Path("/Storage/Pictures/Astronomy/2024-06-24 M8 and M20/LIGHT/") 
OUTPUT_XLSX = Path("multi_model_results.xlsx")
MODELS_DIR = Path("/Storage/Files/practicalML/gitlab/practicalml/training_data")
CLASS_NAMES = ['focus', 'good', 'tracking', 'wind'] 
STARS_TO_EXTRACT = 10
CUTOUT_SIZE = 224

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
# --- Folder Selection ---
try:
    print(f"Please select input folder (Defaulting to {DEFAULT_INPUT_DIR})...")
    root = tk.Tk()
    root.withdraw()
    selected_dir = filedialog.askdirectory(initialdir=DEFAULT_INPUT_DIR, title="Select Input Folder")
    
    if selected_dir:
        INPUT_DIR = Path(selected_dir)
    else:
        print("No folder selected. Using default.")
        INPUT_DIR = DEFAULT_INPUT_DIR
except Exception as e:
    print(f"GUI Selection failed ({e}). Using default path.")
    INPUT_DIR = DEFAULT_INPUT_DIR
    
print(f"Processing Images from: {INPUT_DIR}")
if not INPUT_DIR.exists():
    print("WARNING: Input directory does not exist!")

Please select input folder (Defaulting to /Storage/Pictures/Astronomy/2024-06-24 M8 and M20/LIGHT)...
Processing Images from: /Storage/Pictures/Astronomy/2024-06-24 M8 and M20/LIGHT


In [3]:
# --- Model Factory ---
def create_model(model_name, num_classes=4):
    if model_name == 'MobileNetV2':
        model = models.mobilenet_v2(weights=None)
        model.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        model.classifier[1] = nn.Linear(model.last_channel, num_classes)
        
    elif model_name == 'EfficientNetV2':
        model = models.efficientnet_v2_s(weights=None)
        model.features[0][0] = nn.Conv2d(1, 24, kernel_size=3, stride=2, padding=1, bias=False)
        model.classifier[1] = nn.Linear(1280, num_classes)
        
    elif model_name == 'GoogLeNet':
        model = models.googlenet(weights=None, aux_logits=False, init_weights=True)
        model.conv1.conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        model.fc = nn.Linear(1024, num_classes)
        
    elif model_name == 'ResNet18':
        model = models.resnet18(weights=None)
        model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        model.fc = nn.Linear(512, num_classes)
        
    elif model_name == 'ShuffleNetV2':
        model = models.shufflenet_v2_x1_0(weights=None)
        model.conv1[0] = nn.Conv2d(1, 24, kernel_size=3, stride=2, padding=1, bias=False)
        model.fc = nn.Linear(1024, num_classes)
        
    elif model_name == 'DenseNet121':
        model = models.densenet121(weights=None)
        model.features.conv0 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
        
    elif model_name == 'VGG16':
        model = models.vgg16(weights=None)
        model.features[0] = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1)
        model.classifier[6] = nn.Linear(4096, num_classes)
        
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return model

# --- Load All Models ---
active_models = {}
model_list = ['GoogLeNet', 'EfficientNetV2', 'MobileNetV2', 'ResNet18', 'ShuffleNetV2', 'DenseNet121', 'VGG16']

print(f"Searching for models in {MODELS_DIR}...")
for name in model_list:
    weights_path = MODELS_DIR / f"final_cnn_{name}_best.pth"
    if weights_path.exists():
        try:
            model = create_model(name, len(CLASS_NAMES))
            model.load_state_dict(torch.load(weights_path, map_location=device))
            model.to(device)
            model.eval()
            active_models[name] = model
            print(f"  [LOADED] {name}")
        except Exception as e:
            print(f"  [FAILED] {name}: {e}")
    else:
        print(f"  [MISSING] {name} weights not found")

if not active_models:
    print("!!! NO MODELS LOADED !!!")

Searching for models in /Storage/Files/practicalML/gitlab/practicalml/training_data...
  [LOADED] GoogLeNet
  [LOADED] EfficientNetV2
  [LOADED] MobileNetV2
  [LOADED] ResNet18
  [LOADED] ShuffleNetV2
  [LOADED] DenseNet121
  [LOADED] VGG16


In [4]:
# --- Data Extraction ---
def extract_image_data(file_path, cutout_size=224, stars_to_extract=10):
    try:
        try: img = xisf.XISF.read(str(file_path), image_metadata={})
        except: img = xisf.XISF.read(str(file_path))
        raw_data = np.asarray(img)
        
        if raw_data.max() <= 1.0: raw_uint16 = np.clip(raw_data * 65535.0, 0, 65535).astype(np.uint16)
        else: raw_uint16 = np.clip(raw_data, 0, 65535).astype(np.uint16)

        if raw_uint16.ndim == 3:
             if raw_uint16.shape[0] == 1: raw_uint16 = raw_uint16[0]
             elif raw_uint16.shape[2] == 1: raw_uint16 = raw_uint16[:, :, 0]
        
        if raw_uint16.ndim == 2: rgb_image = cv2.cvtColor(raw_uint16, cv2.COLOR_BAYER_BG2BGR)
        elif raw_uint16.ndim == 3 and raw_uint16.shape[2] == 3: rgb_image = raw_uint16
        else: return (file_path, None, f"Invalid Shape {raw_uint16.shape}")

        height, width = rgb_image.shape[:2]
        small_rgb = cv2.resize(rgb_image, (width // 16, height // 16), interpolation=cv2.INTER_AREA)
        r_med, g_med, b_med = np.median(small_rgb[:,:,2]), np.median(small_rgb[:,:,1]), np.median(small_rgb[:,:,0])
        
        if r_med > 1e-6 and g_med > 1e-6 and b_med > 1e-6:
            r_scale, b_scale = g_med / r_med, g_med / b_med
            b, g, r = cv2.split(rgb_image)
            r = np.clip(r.astype(np.float32) * r_scale, 0, 65535).astype(np.uint16)
            b = np.clip(b.astype(np.float32) * b_scale, 0, 65535).astype(np.uint16)
            rgb_balanced = cv2.merge([b, g, r])
        else: rgb_balanced = rgb_image

        gray_data = cv2.cvtColor(rgb_balanced, cv2.COLOR_BGR2GRAY)
        mean, median, std = sigma_clipped_stats(gray_data.astype(np.float32), sigma=3.0)
        daofind = DAOStarFinder(fwhm=5.0, threshold=8.0*std)
        sources = daofind(gray_data.astype(np.float32) - median)
        if sources is None or len(sources) == 0: return (file_path, None, "no_stars")
        sources.sort('peak'); sources.reverse()
        
        cutouts = []
        bg_float = rgb_balanced.astype(np.float32) / 65535.0
        half = cutout_size // 2
        
        for star in sources:
            x, y = int(star['xcentroid']), int(star['ycentroid'])
            if x < half or x > width - half or y < half or y > height - half: continue
            crop = bg_float[y-half:y+half, x-half:x+half, :]
            b, g, r = cv2.split(crop)
            cutouts.extend([r[...,np.newaxis], g[...,np.newaxis], b[...,np.newaxis]])
            if len(cutouts) >= stars_to_extract * 3: break
            
        return (file_path, np.array(cutouts) if cutouts else None, "success" if cutouts else "no_valid_cutouts")
    except Exception as e: return (file_path, None, f"error: {e}")

In [5]:
# --- Inference & Excel Generation ---
results_data = []
val_transform = transforms.Compose([transforms.ToTensor(), transforms.Resize((224, 224))])
files = list(INPUT_DIR.glob("*.xisf"))

if active_models:
    with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
        future_to_file = {executor.submit(extract_image_data, f): f for f in files}
        for future in as_completed(future_to_file):
            fpath = future_to_file[future]
            _, cutouts, status = future.result()
            row = {'Filename': fpath.name, 'Ground_Truth': "", 'Agreement': "0/0", 'Disagreement': False}
            
            if status != 'success':
                row['Status'] = status
                results_data.append(row)
                continue
                
            batch = torch.stack([val_transform(c) for c in cutouts]).to(device)
            file_votes = []
            
            for name, model in active_models.items():
                with torch.no_grad():
                    out = model(batch)
                    if isinstance(out, tuple): out = out[0]
                    _, preds = torch.max(out, 1)
                    votes = [CLASS_NAMES[p] for p in preds.cpu().numpy()]
                    winner, count = Counter(votes).most_common(1)[0] if votes else ("error", 0)
                    row[f"{name}_Pred"] = winner
                    row[f"{name}_Conf"] = float(count / len(votes))
                    file_votes.append(winner)
            
            maj_winner, maj_count = Counter(file_votes).most_common(1)[0]
            row['Ground_Truth'] = maj_winner # Default to Majority
            row['Agreement'] = f"{maj_count}/{len(active_models)}"
            row['Disagreement'] = (maj_count < len(active_models))
            print(f"Processed {fpath.name} -> {maj_winner}")
            results_data.append(row)

    if results_data:
        df = pd.DataFrame(results_data)
        # Ordering
        base_cols = ['Filename', 'Ground_Truth', 'Agreement', 'Disagreement']
        model_cols = []
        for m in model_list:
            if m in active_models:
                model_cols.extend([f"{m}_Pred", f"{m}_Conf"])
        final_cols = base_cols + [c for c in df.columns if c not in base_cols and c not in model_cols] + model_cols
        df = df[final_cols]
        
        # Save raw first
        df.to_excel(OUTPUT_XLSX, index=False)
        
        # OpenPyXL Formatting
        wb = openpyxl.load_workbook(OUTPUT_XLSX)
        ws = wb.active
        ws.insert_rows(1, 2) # Insert 2 header rows
        
        # Add Headers for Metrics
        ws['A1'] = "Metrics"
        ws['A2'] = "Weighted Score (Sum Conf)"
        ws['A3'] = "Accuracy %"
        
        # Define Data Range (Starting Row 5, up to max)
        max_r = ws.max_row
        gt_col = 'B' # Ground Truth
        
        # Iterate columns to find Model Preds
        # Structure: Pred, Conf, Pred, Conf...
        # We want to add formulas above Pred columns
        header_row = 4 # Use 1-based index, headers are now at row 3 (if inserted 2?)
        # Originally headers at 1. Insert 2 -> Headers at 3. Data at 4.
        # Wait, insert_rows(1, 2) pushes Row 1 to Row 3. So Headers are Row 3.
        
        for col in range(1, ws.max_column + 1):
            header_val = ws.cell(row=3, column=col).value
            if header_val and "_Pred" in str(header_val):
                # Found a prediction column
                col_letter = get_column_letter(col)
                conf_col_letter = get_column_letter(col + 1)
                
                # Formula: Accuracy
                # =SUMPRODUCT(--(C4:C99=$B4:$B99))/COUNTA($B4:$B99)
                acc_formula = f"=SUMPRODUCT(--({col_letter}4:{col_letter}{max_r}=${gt_col}4:${gt_col}{max_r}))/COUNTA(${gt_col}4:${gt_col}{max_r})"
                ws.cell(row=3, column=col).font = Font(bold=True) # Bold Header
                ws.cell(row=3, column=col).value = ws.cell(row=3, column=col).value # Keep name
                ws.cell(row=2, column=col).value = acc_formula # Put Acc here for now
                # Actually request was "two cells above each model".
                
                # Weighted: Sum of Confidence where Pred == GT
                # =SUMPRODUCT(({col}4:{col}N=$B4:$BN) * {conf}4:{conf}N)
                weighted_formula = f"=SUMPRODUCT(({col_letter}4:{col_letter}{max_r}=${gt_col}4:${gt_col}{max_r})*{conf_col_letter}4:{conf_col_letter}{max_r})"
                
                ws.cell(row=1, column=col).value = weighted_formula
                ws.cell(row=2, column=col).value = acc_formula
                
                # Conditional Formatting for Mismatch
                # If Cell != Ground Truth, Highlight Red
                red_fill = PatternFill(start_color="FFCCCC", end_color="FFCCCC", fill_type="solid")
                dxf = openpyxl.styles.differential.DifferentialStyle(fill=red_fill)
                rule = openpyxl.formatting.rule.Rule(type="expression", dxf=dxf, stopIfTrue=True)
                rule.formula = [f"{col_letter}4<>${gt_col}4"] # e.g. E4 <> $B4
                ws.conditional_formatting.add(f"{col_letter}4:{col_letter}{max_r}", rule)

        wb.save(OUTPUT_XLSX)
        print(f"Done! Results saved to {OUTPUT_XLSX}")


WARNING
: NoDetectionsWarning: No sources were found. [photutils.detection.daofinder]

Processed 2024-06-25_03-49-43__5.20_300.00s_0056.xisf -> tracking
Processed 2024-06-25_03-17-26__5.20_300.00s_0050.xisf -> wind
Processed 2024-06-25_03-39-38__5.20_300.00s_0054.xisf -> tracking
Processed 2024-06-25_03-04-41__5.30_300.00s_0048.xisf -> wind


Processed 2024-06-25_00-57-02__5.50_300.00s_0024.xisf -> good
Processed 2024-06-25_01-41-33__5.50_300.00s_0032.xisf -> wind
Processed 2024-06-25_04-30-19__5.20_300.00s_0063.xisf -> tracking
Processed 2024-06-24_23-58-55__5.70_300.00s_0013.xisf -> good
Processed 2024-06-25_04-15-12__5.20_300.00s_0060.xisf -> tracking
Processed 2024-06-25_00-14-42__5.60_300.00s_0016.xisf -> good
Processed 2024-06-24_23-16-57__5.80_300.00s_0005.xisf -> good
Processed 2024-06-24_23-11-56__5.80_300.00s_0004.xisf -> good
Processed 2024-06-25_02-06-42__5.40_300.00s_0037.xisf -> wind
Processed 2024-06-25_00-19-44__5.60_300.00s_0017.xisf -> good
Processed 2024-06-24_22-51-32__5.80_300.00s_0000.xisf -> good
Processed 2024-06-25_00-51-18__5.60_300.00s_0023.xisf -> good
Processed 2024-06-25_01-31-28__5.50_300.00s_0030.xisf -> wind
Processed 2024-06-25_01-19-18__5.50_300.00s_0028.xisf -> wind
Processed 2024-06-25_02-39-33__5.30_300.00s_0043.xisf -> wind
Processed 2024-06-25_01-09-12__5.60_300.00s_0026.xisf -> track